In [10]:
import polars as pl
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [11]:
df = pl.read_csv("../hospital_readmission_dataset.csv")
df.head()

patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
str,str,str,i64,str,str,str,i64,i64,str,i64,i64,i64,str,str,f64,i64
"""P00001""","""2022-04-14""","""Spring""",66,"""Male""","""South""","""Diabetes""",5,6,"""Interventional""",8,6,1,"""Medicare""","""Home Health""",0.92,1
"""P00002""","""2021-09-19""","""Fall""",55,"""Male""","""South""","""Diabetes""",4,6,"""Interventional""",6,4,3,"""Private""","""Home Health""",0.88,1
"""P00003""","""2023-04-12""","""Spring""",69,"""Female""","""West""","""Hypertension""",6,8,"""Medical""",9,6,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00004""","""2023-08-14""","""Summer""",83,"""Male""","""South""","""Stroke""",6,11,"""Medical""",11,4,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00005""","""2021-11-05""","""Fall""",54,"""Female""","""North""","""Stroke""",4,10,"""Medical""",6,2,1,"""Uninsured""","""Home Health""",0.83,1


In [12]:
df.describe()

statistic,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
str,str,str,str,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,f64,f64
"""count""","""8000""","""8000""","""8000""",8000.0,"""8000""","""8000""","""8000""",8000.0,8000.0,"""8000""",8000.0,8000.0,8000.0,"""8000""","""8000""",8000.0,8000.0
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,"""0""","""0""",0.0,0.0
"""mean""",null,null,null,57.411625,null,null,null,4.3185,7.789125,null,7.47575,3.638125,1.5725,null,null,0.7769375,0.772875
"""std""",null,null,null,16.724388,null,null,null,1.358046,1.930252,null,2.287917,1.630415,0.89196,null,null,0.219885,0.419
"""min""","""P00001""","""2021-01-01""","""Fall""",18.0,"""Female""","""Central""","""Appendicitis""",1.0,3.0,"""Conservative""",2.0,0.0,0.0,"""Medicaid""","""Home""",0.07,0.0
"""25%""",null,null,null,46.0,null,null,null,3.0,6.0,null,6.0,2.0,1.0,null,null,0.63,1.0
"""50%""",null,null,null,57.0,null,null,null,4.0,8.0,null,8.0,4.0,1.0,null,null,0.86,1.0
"""75%""",null,null,null,69.0,null,null,null,5.0,9.0,null,9.0,4.0,2.0,null,null,0.97,1.0
"""max""","""P08000""","""2023-12-31""","""Winter""",95.0,"""Male""","""West""","""Stroke""",10.0,15.0,"""Surgical""",18.0,10.0,5.0,"""Uninsured""","""Skilled Nursing""",0.97,1.0


In [13]:
X = df.drop(['label'])
y = df['label']

# Define column groups
numerical_cols = [
    'age', 'comorbidities_count', 'length_of_stay',
    'medications_count', 'followup_visits_last_year',
    'prev_readmissions', 'readmission_risk_score'
]
categorical_cols = [
    'season', 'gender', 'region', 'primary_diagnosis',
    'treatment_type', 'insurance_type', 'discharge_disposition'
]


In [14]:
# Split Test and Training data
# Convert polars DataFrame and Series to pandas DataFrame and Series
X_train, X_test, y_train, y_test = train_test_split(X.to_pandas(),
                                                    y.to_pandas(),
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

In [15]:
pl.DataFrame(X_train).describe()

statistic,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score
str,str,str,str,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,f64
"""count""","""6400""","""6400""","""6400""",6400.0,"""6400""","""6400""","""6400""",6400.0,6400.0,"""6400""",6400.0,6400.0,6400.0,"""6400""","""6400""",6400.0
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,"""0""","""0""",0.0
"""mean""",null,null,null,57.463125,null,null,null,4.3225,7.7996875,null,7.482031,3.632969,1.57375,null,null,0.776792
"""std""",null,null,null,16.847121,null,null,null,1.366921,1.930144,null,2.294463,1.625841,0.899479,null,null,0.220791
"""min""","""P00001""","""2021-01-01""","""Fall""",18.0,"""Female""","""Central""","""Appendicitis""",1.0,3.0,"""Conservative""",2.0,0.0,0.0,"""Medicaid""","""Home""",0.07
"""25%""",null,null,null,46.0,null,null,null,3.0,6.0,null,6.0,2.0,1.0,null,null,0.63
"""50%""",null,null,null,57.0,null,null,null,4.0,8.0,null,8.0,4.0,1.0,null,null,0.86
"""75%""",null,null,null,69.0,null,null,null,5.0,9.0,null,9.0,4.0,2.0,null,null,0.97
"""max""","""P08000""","""2023-12-31""","""Winter""",95.0,"""Male""","""West""","""Stroke""",10.0,15.0,"""Surgical""",18.0,10.0,5.0,"""Uninsured""","""Skilled Nursing""",0.97


In [16]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', RobustScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Fit on train only, transform both
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

In [17]:
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

['num__age' 'num__comorbidities_count' 'num__length_of_stay'
 'num__medications_count' 'num__followup_visits_last_year'
 'num__prev_readmissions' 'num__readmission_risk_score' 'cat__season_Fall'
 'cat__season_Spring' 'cat__season_Summer' 'cat__season_Winter'
 'cat__gender_Female' 'cat__gender_Male' 'cat__region_Central'
 'cat__region_East' 'cat__region_North' 'cat__region_South'
 'cat__region_West' 'cat__primary_diagnosis_Appendicitis'
 'cat__primary_diagnosis_COPD' 'cat__primary_diagnosis_Diabetes'
 'cat__primary_diagnosis_Fracture' 'cat__primary_diagnosis_Heart Failure'
 'cat__primary_diagnosis_Hypertension' 'cat__primary_diagnosis_Influenza'
 'cat__primary_diagnosis_Kidney Disease'
 'cat__primary_diagnosis_Pneumonia' 'cat__primary_diagnosis_Sepsis'
 'cat__primary_diagnosis_Stroke' 'cat__treatment_type_Conservative'
 'cat__treatment_type_Interventional' 'cat__treatment_type_Medical'
 'cat__treatment_type_Surgical' 'cat__insurance_type_Medicaid'
 'cat__insurance_type_Medicare' 'cat__i

In [21]:
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_train_df.describe()

,num__age,num__comorbidities_count,num__length_of_stay,num__medications_count,num__followup_visits_last_year,num__prev_readmissions,num__readmission_risk_score,cat__season_Fall,cat__season_Spring,cat__season_Summer,...,cat__treatment_type_Medical,cat__treatment_type_Surgical,cat__insurance_type_Medicaid,cat__insurance_type_Medicare,cat__insurance_type_Private,cat__insurance_type_Uninsured,cat__discharge_disposition_Home,cat__discharge_disposition_Home Health,cat__discharge_disposition_Rehab,cat__discharge_disposition_Skilled Nursing
count,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,...,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000
mean,0.020136,0.161250,-0.066771,-0.172656,-0.183516,0.573750,-0.244729,0.249219,0.249531,0.247656,...,0.449688,0.245625,0.201250,0.317969,0.381406,0.099375,0.040313,0.754531,0.006406,0.198750
std,0.732484,0.683461,0.643381,0.764821,0.812920,0.899479,0.649384,0.432595,0.432776,0.431685,...,0.497501,0.430491,0.400966,0.465724,0.485770,0.299188,0.196707,0.430398,0.079788,0.399091
min,-1.695652,-1.500000,-1.666667,-2.000000,-2.000000,-1.000000,-2.323529,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.478261,-0.500000,-0.666667,-0.666667,-1.000000,0.000000,-0.676471,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,0.521739,0.500000,0.333333,0.333333,0.000000,1.000000,0.323529,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,1.652174,3.000000,2.333333,3.333333,3.000000,4.000000,0.323529,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
